# Offline Latent PCA and UMAP

This notebook analyzes offline latent trajectories collected with `research/scripts/collect_latents_offline.py`.

It does four things:
1. loads `metadata.jsonl` and the Parquet latent buffer,
2. builds trajectory-level features from the reasoning span or full response,
3. visualizes successful vs failed trajectories in PCA and UMAP,
4. optionally plots a few token-level trajectories as 2D paths.

Expected input layout:
- `OUTPUT_DIR/metadata.jsonl`
- `OUTPUT_DIR/buffer/manifest.json`
- `OUTPUT_DIR/buffer/shards/*.parquet`

In [1]:
from pathlib import Path
import json
import random

import matplotlib.pyplot as plt
import numpy as np
import torch

from research.buffer import ParquetLatentBuffer
from research.latent.codec import deserialize_latent_tensor

# Edit this if you want a different run.
OUTPUT_DIR = Path('/workspace/outputs/qwen3_4b_math500_latents_pilot_4096_500')
# OUTPUT_DIR = Path('/workspace/outputs/tulu3_math500_latents_pilot_4096')
# OUTPUT_DIR = Path('/workspace/outputs/qwen3_4b_math500_latents_pilot_4096')
OUTPUT_DIR = Path('/workspace/outputs/tulu3_math500_latents_pilot_4096_500')
USE_REASONING_SPAN = True
MAX_ROWS = None
RANDOM_SEED = 1
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

def _load_jsonl(path: Path):
    rows = []
    with path.open(encoding='utf-8') as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

metadata_path = OUTPUT_DIR / 'metadata.jsonl'
buffer_dir = OUTPUT_DIR / 'buffer'
assert metadata_path.exists(), f'missing metadata: {metadata_path}'
assert buffer_dir.exists(), f'missing buffer: {buffer_dir}'

metadata_rows = _load_jsonl(metadata_path)
if MAX_ROWS is not None:
    metadata_rows = metadata_rows[:MAX_ROWS]

metadata_by_uid = {row['uid']: row for row in metadata_rows}
print(f'Loaded {len(metadata_rows)} metadata rows from {metadata_path}')
print(f'Buffer dir: {buffer_dir}')

In [2]:
# Load latent rows directly from the Parquet buffer.
buffer = ParquetLatentBuffer(
    root_dir=str(buffer_dir),
    shard_max_samples=256,
    compression='zstd',
    max_disk_gb=1000.0,
)

rows = list(buffer.iter_rows())
rows = [row for row in rows if row['uid'] in metadata_by_uid]
rows.sort(key=lambda row: metadata_by_uid[row['uid']]['step'])
print(f'Loaded {len(rows)} latent rows from buffer')
print('Buffer stats:', buffer.get_stats())

In [3]:
def extract_sequence_for_analysis(row, meta, use_reasoning_span=True):
    seq = deserialize_latent_tensor(
        row['latent_blob'],
        seq_len=int(row['response_length']),
        hidden_dim=int(row['hidden_dim']),
        dtype=str(row['dtype']),
    ).float()

    start = 0
    end = seq.shape[0]
    if use_reasoning_span and meta.get('think_token_start') is not None and meta.get('think_token_end') is not None:
        start = int(meta['think_token_start'])
        end = int(meta['think_token_end'])
        start = max(0, min(start, seq.shape[0]))
        end = max(start + 1, min(end, seq.shape[0]))
    return seq[start:end]

samples = []
for row in rows:
    meta = metadata_by_uid[row['uid']]
    seq = extract_sequence_for_analysis(row, meta, use_reasoning_span=USE_REASONING_SPAN)
    pooled_mean = seq.mean(dim=0)
    pooled_last = seq[-1]
    step_deltas = seq[1:] - seq[:-1] if seq.shape[0] > 1 else torch.zeros_like(seq[:1])
    path_length = step_deltas.norm(dim=-1).sum().item() if seq.shape[0] > 1 else 0.0
    mean_norm = seq.norm(dim=-1).mean().item()

    samples.append({
        'uid': row['uid'],
        'step': meta['step'],
        'success': bool(meta['success']),
        'score_accuracy': float(meta['score_accuracy']),
        'response_length': int(meta['response_length']),
        'analysis_length': int(seq.shape[0]),
        'has_think_tags': bool(meta.get('has_think_tags', False)),
        'question': meta['question'],
        'response': meta['response'],
        'sequence': seq,
        'mean_feature': pooled_mean.numpy(),
        'last_feature': pooled_last.numpy(),
        'path_length': path_length,
        'mean_norm': mean_norm,
    })

print(f'Prepared {len(samples)} samples')
print('Success count:', sum(int(s['success']) for s in samples))
print('Failure count:', sum(int(not s['success']) for s in samples))

In [4]:
def summarize_group(values, mask):
    picked = [v for v, m in zip(values, mask) if m]
    if not picked:
        return {'n': 0, 'mean': None, 'std': None}
    arr = np.asarray(picked, dtype=np.float64)
    return {'n': int(arr.size), 'mean': float(arr.mean()), 'std': float(arr.std())}

success_mask = [s['success'] for s in samples]
print('Response length stats')
print('  success:', summarize_group([s['response_length'] for s in samples], success_mask))
print('  failure:', summarize_group([s['response_length'] for s in samples], [not x for x in success_mask]))
print('Analysis length stats')
print('  success:', summarize_group([s['analysis_length'] for s in samples], success_mask))
print('  failure:', summarize_group([s['analysis_length'] for s in samples], [not x for x in success_mask]))
print('Path length stats')
print('  success:', summarize_group([s['path_length'] for s in samples], success_mask))
print('  failure:', summarize_group([s['path_length'] for s in samples], [not x for x in success_mask]))
print('Mean latent norm stats')
print('  success:', summarize_group([s['mean_norm'] for s in samples], success_mask))
print('  failure:', summarize_group([s['mean_norm'] for s in samples], [not x for x in success_mask]))

In [5]:
def pca_project(x, n_components=3):
    x = np.asarray(x, dtype=np.float32)
    x_centered = x - x.mean(axis=0, keepdims=True)
    u, s, vt = np.linalg.svd(x_centered, full_matrices=False)
    coords = x_centered @ vt[:n_components].T
    explained = (s ** 2) / max(1, x.shape[0] - 1)
    explained_ratio = explained / explained.sum()
    return coords, explained_ratio[:n_components]

mean_features = np.stack([s['mean_feature'] for s in samples], axis=0)
last_features = np.stack([s['last_feature'] for s in samples], axis=0)
labels = np.asarray([int(s['success']) for s in samples])

mean_pca, mean_var = pca_project(mean_features)
last_pca, last_var = pca_project(last_features)

print('Mean-feature PCA explained variance:', mean_var)
print('Last-token PCA explained variance:', last_var)


In [6]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401


def plot_binary_scatter_3d(coords, labels, title, var_ratio=None):
    fig = plt.figure(figsize=(8, 7))
    ax = fig.add_subplot(111, projection='3d')
    failure = labels == 0
    success = labels == 1
    if failure.any():
        ax.scatter(coords[failure, 0], coords[failure, 1], coords[failure, 2], s=36, alpha=0.8, label='failure', c='#d1495b')
    if success.any():
        ax.scatter(coords[success, 0], coords[success, 1], coords[success, 2], s=36, alpha=0.8, label='success', c='#2a9d8f')
    ax.set_title(title)
    if var_ratio is None or len(var_ratio) < 3:
        ax.set_xlabel('PC1')
        ax.set_ylabel('PC2')
        ax.set_zlabel('PC3')
    else:
        ax.set_xlabel(f'PC1 ({var_ratio[0] * 100:.1f}% var)')
        ax.set_ylabel(f'PC2 ({var_ratio[1] * 100:.1f}% var)')
        ax.set_zlabel(f'PC3 ({var_ratio[2] * 100:.1f}% var)')
    ax.legend()
    plt.show()

plot_binary_scatter_3d(
    mean_pca,
    labels,
    title=f'Mean-pooled trajectory PCA ({"reasoning span" if USE_REASONING_SPAN else "full response"})',
    var_ratio=mean_var,
)

plot_binary_scatter_3d(
    last_pca,
    labels,
    title=f'Last-token trajectory PCA ({"reasoning span" if USE_REASONING_SPAN else "full response"})',
    var_ratio=last_var,
)


## UMAP

This cell uses `umap-learn` if it is available in your environment. Inside the Singularity image it may already be installed. If not, install it there and rerun the cell.

In [7]:
try:
    import umap
    have_umap = True
except Exception as exc:
    have_umap = False
    print('UMAP is unavailable:', exc)

if have_umap:
    reducer = umap.UMAP(
        n_neighbors=min(15, max(2, len(samples) - 1)),
        min_dist=0.1,
        metric='euclidean',
        random_state=RANDOM_SEED,
    )
    mean_umap = reducer.fit_transform(mean_features)
    plot_binary_scatter(mean_umap, labels, title='Mean-pooled trajectory UMAP', xlabel='UMAP1', ylabel='UMAP2')

    reducer_last = umap.UMAP(
        n_neighbors=min(15, max(2, len(samples) - 1)),
        min_dist=0.1,
        metric='euclidean',
        random_state=RANDOM_SEED,
    )
    last_umap = reducer_last.fit_transform(last_features)
    plot_binary_scatter(last_umap, labels, title='Last-token trajectory UMAP', xlabel='UMAP1', ylabel='UMAP2')

## Token-level trajectory paths

The next cell projects token-level latent trajectories into 2D PCA and overlays a few successful and failed examples as paths.
This is useful when class-level scatter is not enough and you want to inspect how latent states evolve over the reasoning process.

In [8]:
# Build a bounded PCA basis from a random sample of token vectors.
MAX_TOKEN_POINTS = 20000
MAX_PATHS_PER_CLASS = 3

eligible = [s for s in samples if s['sequence'].shape[0] >= 2]
all_tokens = np.concatenate([s['sequence'].numpy() for s in eligible], axis=0)
print('total token points:', all_tokens.shape[0])

if all_tokens.shape[0] > MAX_TOKEN_POINTS:
    idx = np.random.choice(all_tokens.shape[0], size=MAX_TOKEN_POINTS, replace=False)
    token_basis_data = all_tokens[idx]
else:
    token_basis_data = all_tokens

_, token_var = pca_project(token_basis_data)
xb = token_basis_data.astype(np.float32)
xb_centered = xb - xb.mean(axis=0, keepdims=True)
_, _, vt = np.linalg.svd(xb_centered, full_matrices=False)
vt2 = vt[:2].T
mean_all = xb.mean(axis=0, keepdims=True)


def project_sequence(seq):
    return (seq.numpy().astype(np.float32) - mean_all) @ vt2

success_examples = [s for s in samples if s['success'] and s['sequence'].shape[0] >= 2]
failure_examples = [s for s in samples if (not s['success']) and s['sequence'].shape[0] >= 2]

random.shuffle(success_examples)
random.shuffle(failure_examples)
success_examples = success_examples[:MAX_PATHS_PER_CLASS]
failure_examples = failure_examples[:MAX_PATHS_PER_CLASS]

plt.figure(figsize=(8, 7))
for sample in failure_examples:
    coords = project_sequence(sample['sequence'])
    plt.plot(coords[:, 0], coords[:, 1], alpha=0.7, c='#d1495b')
    plt.scatter(coords[0, 0], coords[0, 1], c='#d1495b', marker='x', s=40)
    plt.scatter(coords[-1, 0], coords[-1, 1], c='#d1495b', marker='o', s=40)

for sample in success_examples:
    coords = project_sequence(sample['sequence'])
    plt.plot(coords[:, 0], coords[:, 1], alpha=0.7, c='#2a9d8f')
    plt.scatter(coords[0, 0], coords[0, 1], c='#2a9d8f', marker='x', s=40)
    plt.scatter(coords[-1, 0], coords[-1, 1], c='#2a9d8f', marker='o', s=40)

plt.title('Token-level trajectory paths in PCA space')
plt.xlabel(f'PC1 ({token_var[0] * 100:.1f}% var)')
plt.ylabel(f'PC2 ({token_var[1] * 100:.1f}% var)')
plt.grid(alpha=0.2)
plt.show()


## Inspect individual examples

Use this cell to print a few representative examples after you identify interesting clusters or outliers.

In [9]:
def show_examples(sample_list, n=3):
    for sample in sample_list[:n]:
        print('=' * 120)
        print('uid:', sample['uid'])
        print('success:', sample['success'])
        print('score_accuracy:', sample['score_accuracy'])
        print('response_length:', sample['response_length'])
        print('analysis_length:', sample['analysis_length'])
        print('path_length:', round(sample['path_length'], 4))
        print('question:', sample['question'])
        print('response preview:', sample['response'])
        print()

print('Successful examples')
show_examples([s for s in samples if s['success']], n=2)
print('Failed examples')
show_examples([s for s in samples if not s['success']], n=2)